In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
#Train
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

# ============================ CONFIG ============================
ROOT_PATH = "/content/drive/MyDrive/Anomaly-Videos-Part-1"
WORK_DIR  = "/content/drive/MyDrive/AnomalyProject"
IMG_SIZE = (224, 224)
SEQ_LEN = 32
FRAME_STEP = 4
EMBED_DIM = 512
BATCH_SIZE = 8
EPOCHS = 20

os.makedirs(WORK_DIR, exist_ok=True)
FRAMES_DIR = os.path.join(WORK_DIR, "frames")
EMB_DIR = os.path.join(WORK_DIR, "embeddings")
os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(EMB_DIR, exist_ok=True)

# ============================ 1) LIST VIDEO ============================
def list_videos(root):
    items = []
    for cls in sorted(os.listdir(root)):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        label = "Normal" if cls.lower() == "normal" else "Anomaly"
        for subdir, _, files in os.walk(cls_dir):
            for f in files:
                if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
                    items.append({"path": os.path.join(subdir, f), "label": label})
    return items

videos = list_videos(ROOT_PATH)
print(f"Found {len(videos)} videos")

# ============================ 2) EXTRACT FRAMES (subsample) ============================
def extract_frames(video_path, out_dir, step=FRAME_STEP):
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    idx = 0
    frames = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            frame = cv2.resize(frame, IMG_SIZE)
            cv2.imwrite(os.path.join(out_dir, f"frame_{frames:04d}.jpg"), frame)
            frames += 1
        idx += 1
    cap.release()
    return frames

print("\n=== Extracting frames ===")
for v in tqdm(videos):
    vid_name = os.path.splitext(os.path.basename(v["path"]))[0]
    out = os.path.join(FRAMES_DIR, vid_name)
    if not os.path.exists(out) or len(os.listdir(out)) == 0:
        extract_frames(v["path"], out)

# ============================ 3) CNN EMBEDDING ============================
base = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(EMBED_DIM, activation="relu")(x)
cnn_model = models.Model(base.input, x)
cnn_model.trainable = False

def embed_video(frames_dir, out_npy):
    frames = sorted([os.path.join(frames_dir, f) for f in os.listdir(frames_dir) if f.endswith(".jpg")])
    emb_list = []
    for fp in frames:
        img = image.load_img(fp, target_size=IMG_SIZE)
        arr = image.img_to_array(img)[None, ...]
        arr = preprocess_input(arr)
        emb = cnn_model.predict(arr, verbose=0)
        emb_list.append(emb[0].astype(np.float16))  # float16 để giảm memory
    emb = np.array(emb_list)
    np.save(out_npy, emb)
    return emb

print("\n=== Embedding videos ===")
frame_index = []
for v in tqdm(videos):
    vid_name = os.path.splitext(os.path.basename(v["path"]))[0]
    fdir = os.path.join(FRAMES_DIR, vid_name)
    emb = os.path.join(EMB_DIR, vid_name + ".npy")
    if not os.path.exists(emb):
        embed_video(fdir, emb)
    arr = np.load(emb)
    frame_index.append({
        "video_name": vid_name,
        "emb_path": emb,
        "label": v["label"],
        "n_frames": len(arr)
    })

# ============================ 4) BUILD SEQUENCES (chunk video dài) ============================
def make_seq(emb, seq_len=SEQ_LEN, step=SEQ_LEN):
    seqs = []
    n = len(emb)
    if n < seq_len:
        pad = np.repeat(emb[-1][None,:], seq_len-n, 0)
        seqs.append(np.vstack([emb, pad]))
    else:
        for i in range(0, n - seq_len + 1, step):
            seqs.append(emb[i:i+seq_len])
        # Nếu còn phần dư chưa đủ seq_len, thêm 1 chunk cuối
        if n % seq_len != 0:
            last_chunk = emb[-seq_len:]
            seqs.append(last_chunk)
    return np.array(seqs)

X, y = [], []
for item in frame_index:
    emb = np.load(item["emb_path"])
    seqs = make_seq(emb)
    label = 0 if item["label"].lower()=="normal" else 1
    X.append(seqs)
    y.append(np.full(len(seqs), label))

X = np.vstack(X)
y = np.hstack(y)
print("\nDataset:", X.shape, y.shape)

# ============================ 5) LSTM MODEL ============================
def build_lstm():
    inp = layers.Input(shape=(SEQ_LEN, EMBED_DIM))
    x = layers.Bidirectional(layers.LSTM(128))(inp)  # giảm hidden size
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inp, out)
    model.compile(optimizer=optimizers.Adam(1e-4),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

model = build_lstm()
model.summary()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

ckpt_path = os.path.join(WORK_DIR, "best_lstm.h5")
cb = [
    callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss"),
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb
)

model.save(os.path.join(WORK_DIR, "lstm_final.h5"))
print("\nTraining complete. Models saved in:", WORK_DIR)


Found 130 videos

=== Extracting frames ===


100%|██████████| 130/130 [00:12<00:00, 10.03it/s]



=== Embedding videos ===


100%|██████████| 130/130 [3:02:37<00:00, 84.29s/it] 



Dataset: (3185, 32, 512) (3185,)


...

319/319 ━━━━━━━━━━━━━━━━━━━━ 34s 105ms/step - accuracy: 1.0000 - loss: 7.4458e-04 - val_accuracy: 1.0000 - val_loss: 2.6252e-04
Epoch 6/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 40s 104ms/step - accuracy: 1.0000 - loss: 5.6071e-04 - val_accuracy: 1.0000 - val_loss: 2.6680e-04
Epoch 7/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 1.0000 - loss: 4.3952e-04

319/319 ━━━━━━━━━━━━━━━━━━━━ 32s 100ms/step - accuracy: 1.0000 - loss: 4.3940e-04 - val_accuracy: 1.0000 - val_loss: 7.3778e-05
Epoch 8/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 46s 116ms/step - accuracy: 1.0000 - loss: 2.6716e-04 - val_accuracy: 1.0000 - val_loss: 1.1651e-04
Epoch 9/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 32s 101ms/step - accuracy: 1.0000 - loss: 2.6776e-04 - val_accuracy: 1.0000 - val_loss: 1.5448e-04
Epoch 10/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 32s 100ms/step - accuracy: 1.0000 - loss: 1.7897e-04 - val_accuracy: 1.0000 - val_loss: 1.3361e-04
Epoch 11/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 1.0000 - loss: 1.4269e-04

319/319 ━━━━━━━━━━━━━━━━━━━━ 41s 101ms/step - accuracy: 1.0000 - loss: 1.4270e-04 - val_accuracy: 1.0000 - val_loss: 3.3803e-05
Epoch 12/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 31s 99ms/step - accuracy: 1.0000 - loss: 1.1223e-04 - val_accuracy: 1.0000 - val_loss: 1.2663e-04
Epoch 13/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 34s 107ms/step - accuracy: 1.0000 - loss: 8.0135e-05 - val_accuracy: 1.0000 - val_loss: 1.1233e-04
Epoch 14/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 32s 100ms/step - accuracy: 1.0000 - loss: 7.1065e-05 - val_accuracy: 1.0000 - val_loss: 9.1103e-05
Epoch 15/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 34s 105ms/step - accuracy: 1.0000 - loss: 4.6780e-05 - val_accuracy: 1.0000 - val_loss: 1.1761e-04
Epoch 16/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 1.0000 - loss: 9.7259e-05

319/319 ━━━━━━━━━━━━━━━━━━━━ 39s 100ms/step - accuracy: 1.0000 - loss: 9.7188e-05 - val_accuracy: 1.0000 - val_loss: 6.1495e-06
Epoch 17/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 41s 100ms/step - accuracy: 1.0000 - loss: 3.0047e-05 - val_accuracy: 1.0000 - val_loss: 7.3176e-06
Epoch 18/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 33s 102ms/step - accuracy: 1.0000 - loss: 3.2629e-05 - val_accuracy: 1.0000 - val_loss: 9.7013e-06
Epoch 19/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 36s 113ms/step - accuracy: 1.0000 - loss: 2.5033e-05 - val_accuracy: 1.0000 - val_loss: 1.0358e-05
Epoch 20/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 32s 101ms/step - accuracy: 1.0000 - loss: 2.0845e-05 - val_accuracy: 1.0000 - val_loss: 1.2982e-05



Training complete. Models saved in: /content/drive/MyDrive/AnomalyProject


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input

# ============================ CONFIG ============================
ROOT_PATH = "/content/drive/MyDrive/Anomaly-Videos-Part-1"
WORK_DIR  = "/content/drive/MyDrive/AnomalyProject"
IMG_SIZE = (224, 224)
SEQ_LEN = 32
FRAME_STEP = 4
EMBED_DIM = 512
BATCH_SIZE = 8
EPOCHS = 20

os.makedirs(WORK_DIR, exist_ok=True)
FRAMES_DIR = os.path.join(WORK_DIR, "frames")
EMB_DIR = os.path.join(WORK_DIR, "embeddings")
os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(EMB_DIR, exist_ok=True)

# ============================ 1) LIST VIDEO ============================
def list_videos(root):
    items = []
    for cls in sorted(os.listdir(root)):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        label = "Normal" if cls.lower() == "normal" else "Anomaly"
        for subdir, _, files in os.walk(cls_dir):
            for f in files:
                if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
                    items.append({"path": os.path.join(subdir, f), "label": label})
    return items

videos = list_videos(ROOT_PATH)
print(f"Found {len(videos)} videos")

# ============================ 2) EXTRACT FRAMES ============================
def extract_frames(video_path, out_dir, step=FRAME_STEP):
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    idx, frames = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            frame = cv2.resize(frame, IMG_SIZE)
            cv2.imwrite(os.path.join(out_dir, f"frame_{frames:04d}.jpg"), frame)
            frames += 1
        idx += 1
    cap.release()
    return frames

print("\n=== Extracting frames ===")
for v in tqdm(videos):
    vid_name = os.path.splitext(os.path.basename(v["path"]))[0]
    out = os.path.join(FRAMES_DIR, vid_name)
    if not os.path.exists(out) or len(os.listdir(out)) == 0:
        extract_frames(v["path"], out)

# ============================ 3) CNN EMBEDDING ============================
base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224,224,3))
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(EMBED_DIM, activation="relu")(x)
cnn_model = models.Model(base.input, x)
cnn_model.trainable = False

def embed_video(frames_dir, out_npy):
    frames = sorted([os.path.join(frames_dir, f) for f in os.listdir(frames_dir) if f.endswith(".jpg")])
    emb_list = []
    for fp in frames:
        img = image.load_img(fp, target_size=IMG_SIZE)
        arr = image.img_to_array(img)[None, ...]
        arr = preprocess_input(arr)
        emb = cnn_model.predict(arr, verbose=0)[0].astype(np.float32)

        # === FIX 3: Add slight augmentation ===
        noise = np.random.normal(0, 0.01, emb.shape)
        emb = emb + noise

        emb_list.append(emb.astype(np.float16))

    emb = np.array(emb_list)
    np.save(out_npy, emb)
    return emb

print("\n=== Embedding videos ===")
frame_index = []
for v in tqdm(videos):
    vid_name = os.path.splitext(os.path.basename(v["path"]))[0]
    fdir = os.path.join(FRAMES_DIR, vid_name)
    emb_path = os.path.join(EMB_DIR, vid_name + ".npy")

    if not os.path.exists(emb_path):
        embed_video(fdir, emb_path)

    arr = np.load(emb_path)
    frame_index.append({
        "video_name": vid_name,
        "emb_path": emb_path,
        "label": v["label"],
        "n_frames": len(arr)
    })

# ============================ 4) BUILD SEQUENCES ============================
def make_seq(emb, seq_len=SEQ_LEN, step=SEQ_LEN):
    seqs = []
    n = len(emb)
    if n < seq_len:
        pad = np.repeat(emb[-1][None,:], seq_len-n, 0)
        seqs.append(np.vstack([emb, pad]))
    else:
        for i in range(0, n - seq_len + 1, step):
            seqs.append(emb[i:i+seq_len])
        if n % seq_len != 0:
            seqs.append(emb[-seq_len:])
    return np.array(seqs)

# Build sequence per video
video_seqs = []
video_labels = []

for item in frame_index:
    emb = np.load(item["emb_path"])
    seqs = make_seq(emb)
    label = 0 if item["label"].lower() == "normal" else 1
    video_seqs.append(seqs)
    video_labels.append(label)

# ============================ FIX 1: split theo video ============================
train_idx, val_idx = train_test_split(
    np.arange(len(video_seqs)),
    test_size=0.2,
    stratify=video_labels,
    random_state=42
)

X_train = np.vstack([video_seqs[i] for i in train_idx])
X_val   = np.vstack([video_seqs[i] for i in val_idx])
y_train = np.hstack([np.full(len(video_seqs[i]), video_labels[i]) for i in train_idx])
y_val   = np.hstack([np.full(len(video_seqs[i]), video_labels[i]) for i in val_idx])

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)

# ============================ 5) LSTM MODEL (Fix 2) ============================
def build_lstm():
    inp = layers.Input(shape=(SEQ_LEN, EMBED_DIM))
    x = layers.Bidirectional(layers.LSTM(
        128,
        return_sequences=False,
        dropout=0.3,
        recurrent_dropout=0.3
    ))(inp)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_lstm()
model.summary()

ckpt = os.path.join(WORK_DIR, "best_lstm.keras")
cb = [
    callbacks.ModelCheckpoint(ckpt, save_best_only=True, monitor="val_loss"),
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb
)

model.save(os.path.join(WORK_DIR, "lstm_final.keras"))
print("\nTraining complete. Models saved:", WORK_DIR)


Found 130 videos

=== Extracting frames ===


  2%|▏         | 2/130 [00:05<05:50,  2.74s/it]


KeyboardInterrupt: 

Test video

In [ ]:
#Test video
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

# ========================= CONFIG =========================
IMG_SIZE = (224, 224)
SEQ_LEN = 32
EMBED_DIM = 512

WORK_DIR = "/content/drive/MyDrive/AnomalyProject"
CNN_PATH = None  # CNN là feature extractor dùng chung code trên
LSTM_PATH = f"{WORK_DIR}/best_lstm.keras"

# Load lại CNN Feature Extractor
from tensorflow.keras.applications.efficientnet import EfficientNetB0

base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224,224,3))
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(EMBED_DIM, activation="relu")(x)
cnn_model = models.Model(base.input, x)
cnn_model.trainable = False

# Load LSTM model
lstm_model = models.load_model(LSTM_PATH)
print("Model loaded")


# ========================= 1) Extract frames =========================
def extract_frames_for_test(video_path, step=4):
    frames = []
    cap = cv2.VideoCapture(video_path)
    idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            frame = cv2.resize(frame, IMG_SIZE)
            frames.append(frame)
        idx += 1

    cap.release()
    return np.array(frames)


# ========================= 2) CNN embedding =========================
def embed_frames(frames):
    emb_list = []
    for f in frames:
        arr = np.expand_dims(f, axis=0)
        arr = preprocess_input(arr)
        emb = cnn_model.predict(arr, verbose=0)[0]

        # noise augmentation nhẹ như training
        noise = np.random.normal(0, 0.01, emb.shape)
        emb = emb + noise

        emb_list.append(emb)
    return np.array(emb_list)


# ========================= 3) Create sequences =========================
def make_seq(emb, seq_len=32):
    seqs = []
    n = len(emb)
    if n < seq_len:
        pad = np.repeat(emb[-1][None,:], seq_len-n, 0)
        seqs.append(np.vstack([emb, pad]))
    else:
        for i in range(0, n - seq_len + 1, seq_len):
            seqs.append(emb[i:i+seq_len])

        if n % seq_len != 0:
            seqs.append(emb[-seq_len:])
    return np.array(seqs)


# ========================= 4) Predict video anomaly =========================
def predict_video(video_path):
    print(f"\n=== Testing video: {video_path} ===")

    frames = extract_frames_for_test(video_path)
    print("Frames extracted:", len(frames))

    emb = embed_frames(frames)
    print("Embedding shape:", emb.shape)

    seqs = make_seq(emb)
    print("Sequences:", seqs.shape)

    preds = lstm_model.predict(seqs)
    score = preds.mean()

    print("\nPrediction score:", score)

    if score > 0.61:
        print("🔥 Video classified as: **ANOMALY**")
    else:
        print("✅ Video classified as: **NORMAL**")

    return score


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model loaded


In [ ]:

video_path = "/content/drive/MyDrive/Anomaly-Videos-Part-1/testvideo1.mp4"
predict_video(video_path)


=== Testing video: /content/drive/MyDrive/Anomaly-Videos-Part-1/testvideo1.mp4 ===
Frames extracted: 95


In [ ]:

video_path = "/content/drive/MyDrive/Anomaly-Videos-Part-1/Normal/Normal_Videos_248_x264.mp4"
predict_video(video_path)


=== Testing video: /content/drive/MyDrive/Anomaly-Videos-Part-1/Normal/Normal_Videos_248_x264.mp4 ===
Frames extracted: 285
Embedding shape: (285, 512)
Sequences: (9, 32, 512)
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 824ms/step

Prediction score: 0.47837597
✅ Video classified as: **NORMAL**


np.float32(0.47837597)

In [ ]:
#Bat thuong
video_path = "/content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly/Abuse/Abuse024_x264.mp4"
predict_video(video_path)


=== Testing video: /content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly/Abuse/Abuse024_x264.mp4 ===
Frames extracted: 256
Embedding shape: (256, 512)
Sequences: (8, 32, 512)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step

Prediction score: 0.99639857
🔥 Video classified as: **ANOMALY**


np.float32(0.99639857)

In [ ]:
#Bat thuong
video_path = "/content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly/Abuse/Abuse010_x264.mp4"
predict_video(video_path)


=== Testing video: /content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly/Abuse/Abuse010_x264.mp4 ===
Frames extracted: 284
Embedding shape: (284, 512)
Sequences: (9, 32, 512)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step

Prediction score: 0.81846875
🔥 Video classified as: **ANOMALY**


np.float32(0.81846875)